# Poster Notes — VEP-nAChR (Variant Effect Predictor for nAChRs)

**What this notebook is:** your talking script and cheat-sheet for presenting the poster, explaining every section in plain language, and fielding audience questions — including the two things most people don't know: **the F1 score** and **the "chance" baseline**.

**How to use it:** read the elevator pitch before you present, keep the F1 / baseline sections and the FAQ handy for Q&A, and glance at the "key numbers" table right before you start.

## 1. The one-sentence pitch

> "We built a machine-learning model that, given a mutation in a nicotinic acetylcholine receptor (nAChR), predicts whether that mutation makes the channel **more** active (gain-of-function, GOF) or **less** active (loss-of-function, LOF) — the direction that actually determines treatment, and which existing tools cannot tell you."

## 2. Why this is a real problem (not just "another ML project")

Most variant-effect predictors (VEPs) — **AlphaMissense, PolyPhen-2, CADD, REVEL** — answer one question:

> "Is this mutation **damaging or benign**?"  *(pathogenicity)*

We answer a different, harder question:

> "**Which direction** does the mutation push the channel — **GOF or LOF**?"

**Why the direction matters:**
- A mutation can be damaging in **two opposite ways**: it can over-activate the channel (GOF) or under-activate it (LOF).
- These two cases need **opposite treatments** — a GOF disease needs a channel *blocker*; a LOF disease needs a channel *potentiator*.
- A generic "pathogenic" score only says "bad", not "which way bad", so it cannot guide treatment.

**Analogy:** it's the difference between knowing a car has a fault (pathogenicity) versus knowing whether the brake is stuck *on* (GOF) or stuck *off* (LOF). Fixing each requires the opposite action.

**Our claim:** the first nAChR-specific GOF-vs-LOF predictor.

## 3. Biology in 60 seconds (for a non-biology audience)

- **nAChR** = nicotinic acetylcholine receptor — a protein **channel** in the cell membrane.
- It is **pentameric**: five subunits arranged around a central pore.
- When a messenger (acetylcholine — or nicotine) binds, the channel **opens** and ions (Na⁺, Ca²⁺) flow through, producing an electrical signal.
- Found in the **neuromuscular junction** (muscle contraction) and the **brain**.
- **16 genes** code for the subunits (CHRNA1–7, CHRNA9, CHRNA10; CHRNB1–4; CHRND, CHRNE, CHRNG).

**Terms you must be able to define:**
- **Missense variant** — a single DNA "letter" change that swaps one amino acid for another in the protein.
- **GOF (gain-of-function)** — the mutant channel opens too easily / too much → too much ion flow.
- **LOF (loss-of-function)** — the mutant channel opens poorly / not at all → too little ion flow.
- **NNE (no-net-effect)** — the mutation does not measurably change function.

## 4. Section-by-section walkthrough

### Title
"A Variant Effect Predictor for Nicotinic Acetylcholine Receptors" — we predict effect **direction** specifically for nAChRs, not for all proteins.

### 1. Introduction (top-left)
Sets up the nAChR and the GOF/LOF distinction, then states the gap: existing tools score pathogenicity, not direction, so they can't separate a pathogenic GOF from a pathogenic LOF.

### Challenges (top-right box)
The four honest difficulties (great for answering "what was hard?"):
1. **Small data** — only 717 curated variants → we rely on engineered features, not a big model trained from scratch.
2. **Direction ≠ pathogenicity** — a harder, much less-explored label.
3. **Gene leakage** — variants from the same gene are correlated; a naive train/test split would inflate scores, so we use gene-grouped CV.
4. **Cross-species numbering** — mouse/rat residue positions don't map 1:1 to human; they need alignment.

### 2. Methods (middle-left)
- **Data:** 717 variants · 16 genes · 3 species · labelled GOF/NNE/LOF from the electrophysiology literature.
- **Features:** 75 engineered features in **7 groups** (physicochemical, substitution, positional/gene-identity, conservation, structural core, ESM-2 zero-shot, ESM-2 embeddings).
- **Models & evaluation:** 10 classifiers (LR, SVM, Random Forest, LightGBM, XGBoost, CatBoost, k-NN, MLP, Gaussian NB), tuned with Optuna; nested **gene-grouped** (leave-one-subunit-out) cross-validation.

### 3. Pipeline & Data (Fig 1 + table)
- The pipeline: **curated data → features → classifiers → gene-grouped CV → prediction**.
- **Data creation (highlighted)** — the 717 variants were manually curated from electrophysiology papers; this curation is itself a contribution.
- Table: class split **51% LOF / 23% NNE / 26% GOF**.

### 4. Results (Fig 2)
- **CatBoost 0.508** macro F1 (3-class) and **Random Forest 0.647** (binary), under gene-grouped CV.
- **Above chance, below clinical utility** → honest: the signal is real but modest.
- Conservation + ESM-2 lift 3-class F1 by +0.023 but leave binary unchanged → they help separate "effect" from "no effect", **not** GOF from LOF.

### 5. Feature Analysis (Fig 3)
- **Gene identity dominates**: dropping the positional features costs **−0.078** F1.
- Then substitution (−0.033) and ESM-2 embeddings (−0.020); the rest ≈ neutral.
- Interpretation: **which gene (subunit) a variant is in is the single strongest predictor of its direction.**

### 6. Future Work
1. Scale/diversify the curated dataset with cross-species variants (with principled alignment).
2. Engineer **nAChR-specific mechanism** features (gating energetics, allosteric coupling, subunit interfaces).
3. Build a **gene-conditioned** predictor that uses subunit identity explicitly.
4. Validate against **prospective** electrophysiology.

### References
9 papers: the SOTA VEP methods we compare against (AlphaMissense, PolyPhen-2, CADD, REVEL, MissION, PreMode, LoGoFunc, ESM) plus our ML toolkit (LightGBM).

## 5. F1 score — explained from scratch  (the part most people don't know)

### The two ingredients
- **Precision** = of the variants we called "GOF", how many were *actually* GOF?  (How precise is our "GOF" call?)
  - Precision = TP / (TP + FP)
- **Recall** = of all the *truly* GOF variants, how many did we *catch*?  (How complete is our detection?)
  - Recall = TP / (TP + FN)

*(TP = true positives, FP = false positives, FN = false negatives.)*

**The tension:** predict everything as "GOF" → recall 100% but precision terrible. Predict only the one you're 100% sure of → precision 100% but recall tiny. You need a balance.

### F1 = the balance
F1 is the **harmonic mean** of precision and recall:

> **F1 = 2 · (Precision × Recall) / (Precision + Recall)**

- Range: **0 (worst) to 1 (perfect)**.
- F1 is only high when **both** precision and recall are high.

**Worked example:** 100 variants, 30 are truly GOF. The model flags 20 as GOF; 15 of those are correct (5 false alarms).
- Precision = 15/20 = 0.75
- Recall = 15/30 = 0.50
- F1 = 2·(0.75·0.50)/(0.75+0.50) = 0.75/1.25 = **0.60**

### Why NOT plain accuracy?
Because our classes are **imbalanced** (51% LOF). A dumb model that always predicts "LOF" gets **51% accuracy** yet is useless — accuracy would make it look "better than half". F1 punishes that lazy model hard.

### Macro vs micro F1
- **Macro F1** = compute F1 for **each class separately**, then average the three (GOF, NNE, LOF) **equally**. Every class counts the same, even the small one (NNE = 23%).
- **Micro F1** = pool all predictions into one global count → dominated by the big class.
- We use **macro F1** because we care about getting the smaller classes right too, and because it makes the "chance" baseline clean (next section).

### So is 0.51 "good"?
For a balanced 3-class problem where random guessing = 0.33, a score of **0.51 means clearly above guessing but far from perfect (1.0)**. Call it "modest but real".

## 6. The "chance" baseline — explained  (the other thing most people don't know)

### What is the dashed/dotted line in Fig 2 (0.333 and 0.50)?
It is the **random-guessing floor** — the macro F1 a model with **zero skill** would get by predicting at random.

### Where do those exact numbers come from?
- **3-class** (GOF / NNE / LOF): a random classifier gets macro F1 = 1/3 ≈ **0.333**.
- **Binary** (GOF / LOF): a random classifier gets macro F1 = 1/2 = **0.50**.

**Why 1/(number of classes)?** Because macro F1 treats every class equally. For a random classifier each class has F1 ≈ 1/n, and averaging equally gives 1/n. (This is exactly *why* we use macro F1 — its "no-skill" floor is clean and independent of class imbalance.)

### What the baseline is FOR
1. It's the **"did the model learn anything?"** test. Beat it, or you've learned nothing.
2. It's the **honest reference** for our results: 0.51 and 0.65 are *above* chance → real signal; but not by a lot → the task is genuinely hard.

### A subtlety worth knowing (if someone asks "but 51% accuracy = majority class?")
A majority-class dummy ("always predict LOF") would get **51% accuracy** but only **~0.18 macro F1** — *below* the 0.333 chance line. So the chance line is a meaningful floor that even a lazy classifier falls under; it is not a "gimme".

## 7. Figures explained in detail

### Fig 1 — the pipeline (flowchart)
Five steps, top to bottom:
1. **Curated data** *(outlined/highlighted)* — 717 variants, 16 genes, 3 species, manually curated from electrophysiology literature. This is the foundation and a key contribution.
2. **Features** — 75 engineered, in 7 groups.
3. **Classifiers** — 10 models, Optuna-tuned.
4. **Evaluation** — gene-grouped CV (no leakage).
5. **Prediction** — direction: GOF / NNE / LOF (the orange output box).

### Fig 2 — model comparison (two panels)
- **Top panel — 3-class** (GOF/NNE/LOF), blue bars, sorted best→worst, chance = 0.33.
- **Bottom panel — binary** (GOF/LOF), orange bars, chance = 0.50.
- **Leaders:** CatBoost 0.508 (3-class), Random Forest 0.647 (binary).
- **How to read it:** all 10 models beat chance, but none is dramatically good — the tree ensembles (CatBoost, Random Forest, XGBoost, LightGBM) lead the pack.

### Fig 3 — feature ablation ("what happens if I remove each group?")
- Each bar = the **change in macro F1 when you DELETE that feature group**.
- **Blue (negative)** = removing it *hurts* → that group carries real signal.
- **Orange (positive)** = removing it *helps* (or does nothing) → that group is noise.
- **Gene identity (positional) = −0.078** → the single biggest contributor. Then substitution −0.033 and ESM-2 embeddings −0.020; everything else ≈ 0.
- **Error bars** = spread between Random Forest and CatBoost.
- **Takeaway:** *which gene the variant is in is the strongest predictor of its direction.*

## 8. Anticipated questions (and how to answer)

**Q: What's the difference from AlphaMissense / PolyPhen?**
A: They predict *pathogenicity* (damaging vs benign). We predict *direction* (GOF vs LOF). A pathogenic variant can be GOF or LOF — opposite clinical meaning. We're the first nAChR-specific direction predictor.

**Q: Why only nAChRs?**
A: Direction labels (GOF/LOF) come from electrophysiology experiments, which are labour-intensive. nAChRs are well-studied and medically important (epilepsy, myasthenia, nicotine addiction), so we could curate a high-quality labelled set of 717 variants.

**Q: Isn't 717 a tiny dataset?**
A: Yes — that's exactly why we use engineered features (not a big neural net from scratch), 10 different classifiers, and rigorous gene-grouped CV. And why "scale the data" is future work #1.

**Q: Why do the numbers look low (0.51 / 0.65)?**
A: Because (a) direction-of-effect is a subtle, hard signal, and (b) gene-grouped CV is honest — it doesn't let the model cheat by memorising a gene. Against the chance floor (0.33 / 0.50) the signal is real. See the F1 and baseline sections.

**Q: What does "gene-grouped CV" mean, and why?**
A: Variants from the *same* gene are correlated (same subunit, similar behaviour). A random split would let the model "memorise" which gene a variant is in and look artificially good. So we hold out entire **genes** at test time — the model must generalise to *unseen* genes. It's the honest, hard setting.

**Q: Why is gene identity the biggest feature?**
A: Different subunits have different physiological roles, so a GOF in one subunit tends to map to different effects/diseases than in another. The model learns "which subunit is it?" as a strong prior.

**Q: What are the 75 features, concretely?**
A: 24 physicochemical properties of the amino-acid swap, 3 substitution scores (BLOSUM62, Grantham), 20 positional/gene/species identity flags, 3 conservation scores, 7 structural features (RSA, B-factor, DSSP, …), and 2 + 16 ESM-2 language-model scores/embeddings.

**Q: Is 0.65 (binary) better than 0.51 (3-class)? Can I compare them?**
A: No — different tasks with different chance floors (0.50 vs 0.33). Binary is easier (two classes) and its floor is higher, so the raw numbers aren't comparable.

**Q: What would make it clinically useful?**
A: Higher F1 (ideally > 0.8), validation on prospective/new variants, and calibrated confidence. Right now it's a research result, not a clinical tool — hence "below clinical utility".

**Q: Why not just use a deep model / bigger language model?**
A: We tested ESM-2 at 150M and 650M and the bigger model didn't help — the bottleneck is *labelled data*, not model capacity. Direction signal needs nAChR-specific mechanism, not more generic pathogenicity.

**Q: What's the cross-species angle?**
A: We augment human data with mouse/rat electrophysiology, but residue numbering differs across species, so the data needs alignment — that's future work.

## 9. Key numbers to memorise (quick reference)

| Item | Value |
|---|---|
| Dataset size | 717 variants |
| Genes | 16 nAChR genes |
| Species | 3 (human 507 · rat 138 · mouse 72) |
| Class split | LOF 51% · NNE 23% · GOF 26% |
| Features | 75, in 7 groups |
| Models | 10 classifiers |
| Best 3-class | CatBoost 0.508 macro F1 |
| Best binary | Random Forest 0.647 macro F1 |
| Chance (3-class) | 0.333 |
| Chance (binary) | 0.50 |
| Top feature (ablation) | positional / gene-identity −0.078 |
| CV scheme | nested gene-grouped (leave-one-subunit-out) |

## 10. The 30-second elevator pitch (a script you can say aloud)

"Nicotinic acetylcholine receptors are ion channels encoded by 16 genes, and many disease-causing mutations in them are known. A mutation can make the channel overactive or underactive — gain-of-function or loss-of-function — and those need *opposite* treatments. Existing tools only tell you whether a mutation is damaging, not which direction. We curated 717 experimentally-labelled variants across 16 genes and three species, engineered 75 features, and trained 10 classifiers with gene-grouped cross-validation. Our best models reach a macro F1 of 0.51 for the three-class problem and 0.65 for binary — clearly above chance — and we find that gene identity is the single strongest predictor of effect direction."